遍历每一个文件，然后读取里面的apk的名字和version，最后存为一个apk

写代码,jupyter,我希望遍历这个目录下的所有文件,获取文件名,该文件名为apk_name,然后在该文件下遍历所有文件,如果存在和该文件名高度重合后缀为apk的文件,就把该文件的名字里面的数字记录下来,这个数字作为version,最后输出一个table,里面的列是apk_name和version

In [1]:
import os
import re
import pandas as pd

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
def extract_apk_info(root_dir):
    data = []
    # 遍历根目录及其所有子目录
    for root, dirs, files in os.walk(root_dir):
        # 获取当前文件夹名作为 apk_name (例如 ai.wizely.android)
        apk_name = os.path.basename(root)
        
        # 忽略根目录本身，只处理有包名的子目录
        if not apk_name or apk_name == os.path.basename(root_dir):
            continue
            
        for file in files:
            # 检查是否为 apk 文件，并且文件名包含 apk_name（高度重合）
            if file.endswith('.apk') and apk_name in file:
                # 使用正则表达式提取文件名中的数字序列作为 version
                # 例如从 "ai.wizely.android-546.apk" 提取 "546"
                version_match = re.search(r'-(\d+)\.apk$', file)
                if version_match:
                    version = version_match.group(1)
                    data.append({'apk_name': apk_name, 'version': version})
    
    # 转换为 DataFrame 并去重（防止同个包下有多个相同版本的 config 文件）
    df = pd.DataFrame(data).drop_duplicates()
    return df

In [ ]:
# 设置你的目录路径
target_path = './drive-download-20260228T163116Z-1-002'  # 请根据实际路径修改
result_table = extract_apk_info(target_path)

# 在 Jupyter 中输出表格
result_table.head()

,apk_name,version
0,ai.wizely.android,546
1,app.supercube.mtlkrtn,10107
2,blockpuzzle.wood.sudoku.puzzlegames,4140
3,br.com.mobileasy.comprasparaguai,107
4,cast.video.tool.screenmirroring.casttotv,176


In [5]:
result_table.shape

(28, 2)

写代码,jupyter,我希望遍历这个目录下的所有文件,获取文件名,该文件名为apk_name,然后在该文件下遍历所有文件,如果存在和该文件名高度重合后缀为apk的文件,就把该文件的名字里面的数字记录下来,这个数字作为version,最后输出一个table,里面的列是apk_name和version,
为了确保每个文件夹（apk_name）都能出现在表格中，即使它下面没有符合命名规则的 APK 文件，我们需要调整逻辑：先确定文件夹名，再在其中寻找版本号。如果找了一圈没找到，就给 version 赋一个 None (Null)

In [2]:
import os
import re
import pandas as pd

In [22]:
def extract_apk_info(root_dir):
    data = []
    
    # 1. 获取目标目录下所有的子文件夹名（作为 apk_name）
    # 假设你的目录结构是 100APPS/drive-download.../ai.wizely.android
    for root, dirs, files in os.walk(root_dir):
        # 排除掉包含大量子文件夹的根级目录，只处理具体的包名文件夹
        # 这里通过判断 files 是否存在来定位到最内层的包文件夹
        if not files: 
            continue
            
        apk_name = os.path.basename(root)
        found_version = None
        
        # 2. 在当前文件夹下寻找对应的 APK 文件并提取版本号
        for file in files:
            if file.endswith('.apk') and apk_name in file:
                # 匹配连字符后的数字，例如 "-546.apk" 中的 "546"
                match = re.search(r'-(\d+)\.apk$', file)
                if match:
                    found_version = match.group(1)
                    # break # 找到2版本号即可跳出当前文件的循环
        
                    # 3. 无论是否找到版本号，都记录下这个 apk_name
                    data.append({
                        'apk_name': apk_name,
                        'version': found_version,
                        'ppurl1': None,
                        'wayback1': None,
                    })
    
    # 转换为 DataFrame
    df = pd.DataFrame(data)

    # df.drop(df[apk_name]=="drive-download-20260228T163116Z-1-001")

    return df

In [23]:
# 设置你的目录路径（请根据实际情况修改）
target_path = '1122apk/raw/drive-download-20260305T173717Z-1-002' 
result_table = extract_apk_info(target_path)

# 在 Jupyter 中显示结果
# 使用 fillna('Null') 是为了在表格中更直观地看到缺失值
# result_table.sort_values('apk_name')

In [24]:
result_table.head()

,apk_name,version,ppurl1,wayback1
0,ae.brandsforless.android,412,None,None
1,ae.brandsforless.android,413,None,None
2,air.bg.lan.Monopoli,7000009,None,None
3,air.bg.lan.Monopoli,7000011,None,None
4,air.com.bigwigmedia.hotdogbush,2001179,None,None


In [25]:
result_table.shape

(92, 4)

In [26]:
path = target_path+"/summary.csv"
path

'1122apk/raw/drive-download-20260305T173717Z-1-002/summary.csv'

In [27]:
result_table.to_csv(path)